# TP2 — Réinjection des clusters comme feature
## Fil rouge : Churn Predictor (Session 3)

**Durée estimée : 1h15**

### 🎯 Objectifs d'apprentissage
À la fin de ce TP vous saurez :
- Construire une **feature dérivée d'un clustering** SANS data leakage (fit sur train, predict sur val/test).
- Lancer une **comparaison rigoureuse** "avec" vs "sans" la nouvelle feature, en CV.
- Décider, avec des chiffres, si une nouvelle feature **mérite d'être adoptée**.
- Comprendre quand le clustering apporte de la valeur et quand non.

### 🧠 Le piège du data leakage en non-supervisé

Si on fit le k-means sur **train + val + test ensemble**, les centroïdes connaissent les val/test → le cluster de chaque point val/test est "biaisé optimiste". À l'évaluation, le modèle apparaîtra meilleur qu'il ne sera en prod.

**Règle** : `kmeans.fit(X_train)`, puis `kmeans.predict(X_val)` et `kmeans.predict(X_test)`. Idem pour le préprocesseur scaler+OHE qui précède.


## 0. Setup & rechargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score, confusion_matrix

import mlflow

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


In [ ]:
X_train_fe = pd.read_csv("data/X_train_fe.csv")
X_val_fe   = pd.read_csv("data/X_val_fe.csv")
X_test_fe  = pd.read_csv("data/X_test_fe.csv")
y_train = pd.read_csv("data/y_train.csv")["Churn"]
y_val   = pd.read_csv("data/y_val.csv")["Churn"]
y_test  = pd.read_csv("data/y_test.csv")["Churn"]

print(f"Train: {X_train_fe.shape} | Val: {X_val_fe.shape} | Test: {X_test_fe.shape}")


## 1. Helper : ajouter le cluster comme feature, sans fuite

Cette fonction encapsule **toute la chaîne** :
1. Build préprocesseur (scaler + OHE)
2. Fit sur train
3. Apply sur val et test
4. Fit kmeans sur la version transformée du train
5. Predict cluster sur val et test
6. Retourner les 3 splits enrichis d'une colonne `cluster_id` (de type str pour OHE downstream)

In [ ]:
def add_cluster_feature(X_train, X_val, X_test, n_clusters=4, random_state=RANDOM_STATE):
    """Fit clusterer on train only, predict on the 3 splits.
    Returns a tuple of 3 DataFrames with an extra 'cluster_id' column (str)."""
    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()

    # Build the same preprocessor as before (numeric + categorical)
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])

    # fit_transform on train, transform on val/test
    X_tr_proc = prep.fit_transform(X_train)
    X_va_proc = prep.transform(X_val)
    X_te_proc = prep.transform(X_test)

    # Convert sparse to dense if needed
    if hasattr(X_tr_proc, "toarray"):
        X_tr_proc = X_tr_proc.toarray()
        X_va_proc = X_va_proc.toarray()
        X_te_proc = X_te_proc.toarray()

    # Fit KMeans on X_tr_proc, predict on the 3
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    cluster_train = km.fit_predict(X_tr_proc)
    cluster_val = km.predict(X_va_proc)
    cluster_test = km.predict(X_te_proc)

    # Append column (as string for downstream OHE)
    X_train_out = X_train.assign(cluster_id=cluster_train.astype(str))
    X_val_out   = X_val.assign(cluster_id=cluster_val.astype(str))
    X_test_out  = X_test.assign(cluster_id=cluster_test.astype(str))

    return X_train_out, X_val_out, X_test_out, km


In [ ]:
X_train_c, X_val_c, X_test_c, km_fitted = add_cluster_feature(
    X_train_fe, X_val_fe, X_test_fe, n_clusters=4
)

print(f"With cluster feature: {X_train_c.shape}")
print(f"\ncluster_id distribution in train:")
print(X_train_c["cluster_id"].value_counts().sort_index())
print(f"\ncluster_id distribution in val:")
print(X_val_c["cluster_id"].value_counts().sort_index())


## 2. Comparaison rigoureuse : avec vs sans cluster

On reprend le **best model de S2** (HGBT tuné), on l'évalue en **5-fold CV** sur 2 versions des features :
- **Sans** cluster (le baseline S2)
- **Avec** cluster (la version enrichie)

⚠️ Si vous avez un `artifacts/best_model.joblib` issu de S2, on l'utilisera pour récupérer ses hyperparams. Sinon, on fait un HGBT par défaut.

In [ ]:
def build_pipeline(X):
    """Re-build the same pipeline structure as S2, with HGBT classifier."""
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X.select_dtypes(include="object").columns.tolist()
    
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])
    
    # HGBT with the kind of params we'd typically retain after S2 tuning
    clf = HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_depth=5,
        class_weight="balanced", random_state=RANDOM_STATE,
    )
    return Pipeline([("prep", prep), ("clf", clf)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["f1", "average_precision"]


In [ ]:
# Baseline: WITHOUT cluster feature
print("Running CV WITHOUT cluster feature...")
t0 = time.perf_counter()
pipe_without = build_pipeline(X_train_fe)
cv_without = cross_validate(pipe_without, X_train_fe, y_train, cv=cv, scoring=scoring, n_jobs=2)
print(f"  Done in {time.perf_counter() - t0:.1f}s")


In [ ]:
# WITH cluster feature
print("Running CV WITH cluster feature...")
t0 = time.perf_counter()
pipe_with = build_pipeline(X_train_c)
cv_with = cross_validate(pipe_with, X_train_c, y_train, cv=cv, scoring=scoring, n_jobs=2)
print(f"  Done in {time.perf_counter() - t0:.1f}s")


In [ ]:
summary = pd.DataFrame({
    "metric": ["F1 mean", "F1 std", "PR-AUC mean", "PR-AUC std"],
    "WITHOUT cluster": [
        cv_without["test_f1"].mean(),
        cv_without["test_f1"].std(),
        cv_without["test_average_precision"].mean(),
        cv_without["test_average_precision"].std(),
    ],
    "WITH cluster": [
        cv_with["test_f1"].mean(),
        cv_with["test_f1"].std(),
        cv_with["test_average_precision"].mean(),
        cv_with["test_average_precision"].std(),
    ],
})
summary["delta"] = summary["WITH cluster"] - summary["WITHOUT cluster"]
print(summary.round(4).to_string(index=False))

# Statistical comparison: paired across folds
delta_f1 = cv_with["test_f1"] - cv_without["test_f1"]
print(f"\nPer-fold ΔF1 (WITH - WITHOUT): {delta_f1.round(4).tolist()}")
print(f"Mean ΔF1: {delta_f1.mean():+.4f} | std: {delta_f1.std():.4f}")


### Visualisation : distribution des scores par fold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, label in [(axes[0], "test_f1", "F1"),
                          (axes[1], "test_average_precision", "PR-AUC")]:
    df = pd.DataFrame({
        "WITHOUT cluster": cv_without[metric],
        "WITH cluster": cv_with[metric],
    })
    sns.boxplot(data=df, ax=ax)
    sns.stripplot(data=df, ax=ax, color="black", alpha=0.6, size=8)
    ax.set_ylabel(label)
    ax.set_title(f"{label} per fold")

plt.tight_layout()
plt.show()


### 🧠 Décision : on garde la feature ou pas ?

**Critères de décision** :
- Le gain est-il **statistiquement significatif** (per-fold ΔF1 majoritairement positif) ?
- Le gain est-il **practiquement significatif** (≥ 0.005 F1) ?
- Le coût (complexité ajoutée du préprocesseur clustering en prod) est-il justifié ?

**Cas typiques rencontrés** :
- ΔF1 = +0.001 ± 0.005 → **rejet** (bruit, pas un vrai gain)
- ΔF1 = +0.005 ± 0.003 → **adoption marginale** (à valider en prod)
- ΔF1 = +0.020 ± 0.005 → **adoption claire** (gain solide)
- ΔF1 négatif → **rejet** (le clustering ajoute du bruit)

🧠 **Réflexe Tech Lead** : une feature qui n'apporte rien est une **dette technique** (code à maintenir, modèle à monitorer, point de panne en plus). Refuser une feature qui n'aide pas, c'est aussi du métier.

## 3. Bonus : PCA comme réduction de dimension (avant le modèle)

Et si on **remplaçait** une partie des features par leurs composantes principales ? Idée : compresser sans perdre d'information.

⚠️ **Spoiler** : sur Telco, ça aide rarement les modèles d'arbres (qui sont indifférents à la dimension). Mais c'est utile **conceptuellement** : vous saurez quand l'utiliser (régression linéaire à features très corrélées, par exemple).

In [ ]:
from sklearn.decomposition import PCA

# Build a pipeline that includes PCA (n_components=10) AFTER the preprocessing
# Pipeline: prep → PCA(10) → HGBT
def build_pipeline_with_pca(X, n_components=10):
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X.select_dtypes(include="object").columns.tolist()
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])
    clf = HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_depth=5,
        class_weight="balanced", random_state=RANDOM_STATE,
    )
    return Pipeline([
        ("prep", prep),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ("clf", clf),
    ])

# Run CV with this pipeline and compare to the baseline
pipe_pca = build_pipeline_with_pca(X_train_fe, n_components=10)
cv_pca = cross_validate(pipe_pca, X_train_fe, y_train, cv=cv, scoring=scoring, n_jobs=2)

print(f"WITHOUT PCA: F1 = {cv_without['test_f1'].mean():.4f}")
print(f"WITH PCA(10): F1 = {cv_pca['test_f1'].mean():.4f}")


💡 **Lecture attendue** : PCA va **dégrader** le score, ou le laisser stable. Pourquoi ?
- HGBT n'a pas besoin de réduction de dimension (il gère bien les features corrélées et inutiles).
- PCA mélange l'information : 10 composantes capturent ~80 % de la variance mais perdent 20 %.
- PCA est utile principalement pour **régressions linéaires** sur features très corrélées, ou pour **visualisation**.

🧠 **Réflexe Tech Lead** : ne pas faire de PCA "par réflexe". Demander : *quel problème ça résout ici ?*

## 4. Évaluation finale : la feature cluster sur le test set

Si la décision a été **"on garde la feature cluster"**, on doit revalider sur le test set (qu'on a touché en S2 — on n'utilise PAS le test pour décider, juste pour estimer le score final).

In [ ]:
# Train final model with the cluster feature on full train
final_pipe = build_pipeline(X_train_c)
final_pipe.fit(X_train_c, y_train)

# Evaluate on test
y_test_proba = final_pipe.predict_proba(X_test_c)[:, 1]

# Use the same threshold as S2 (e.g. 0.30) — replace if your S2 threshold differs
S2_THRESHOLD = 0.30
y_test_pred = (y_test_proba >= S2_THRESHOLD).astype(int)

f1_final = f1_score(y_test, y_test_pred)
pr_final = average_precision_score(y_test, y_test_proba)

cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()
gain_final = 55 * tp - 5 * fp

print(f"Final test eval (with cluster feature):")
print(f"  F1: {f1_final:.4f}")
print(f"  PR-AUC: {pr_final:.4f}")
print(f"  Business gain: {gain_final:.0f} €")


## 5. Sauvegarde des artefacts S3

Si la feature cluster est retenue, on sauvegarde **deux artefacts supplémentaires** par rapport à S2 :
- Le **k-means fitté** (utilisé pour calculer le cluster sur des nouveaux clients).
- Le **préprocesseur du clustering** (scaler + OHE utilisé en amont).

🧠 **Pourquoi séparer du best_model.joblib ?** Parce qu'en prod, le pipeline d'inférence sera : *raw client data → preprocessor → kmeans → cluster_id → ColumnTransformer du modèle → HGBT*. Cette pipeline en prod sera **assemblée en S4**.

In [ ]:
Path("artifacts").mkdir(exist_ok=True)
joblib.dump(km_fitted, "artifacts/kmeans.joblib")
joblib.dump(final_pipe, "artifacts/best_model_with_cluster.joblib")

print("✅ Saved:")
print("   artifacts/kmeans.joblib")
print("   artifacts/best_model_with_cluster.joblib")


## 6. Synthèse de la session 3

🎯 **Synthèse** :

> ## Synthèse S3 — Clustering & cluster-as-feature
>
> **Segments client identifiés** (k-means, K=4) :
> - Cluster 0 : Loyalists premium
> - Cluster 1 : Newcomers à risque
> - Cluster 2 : Économes stables
> - Cluster 3 : Heavy users en bascule
>
> **Comparaison avec/sans feature cluster** :
> - F1 sans : ~0.57
> - F1 avec : ~0.59
> - ΔF1 : +0.02 → décision : **on garde** la feature cluster car elle apporte un gain métier.
>
> **Réflexe Tech Lead retenu** :
> - On garde un nouvel input uniquement s’il apporte un gain stable et compréhensible métier.

---

### 🔭 Teaser Session 4 (12 juin)

> *"Le 12 juin, on packaging tout ce qu'on a construit. On va prendre `best_model.joblib` (et `kmeans.joblib` si on garde le cluster), on l'expose en API FastAPI : POST une fiche client → recevoir un score de churn et un cluster. Validation Pydantic, gestion d'erreurs, tests unitaires. C'est le passage du notebook au service web — l'étape critique pour qu'un Tech Lead ait crédibilité auprès des dev."*